# Stage 3 — Unified model comparison (mathematics, same data, same estimator)
M1 van der Linden (2007); M3 Wang & Xu (2015) mixture; M4 diffusion IRT (Molenaar et al., 2015; Wiener FPT via Navarro & Fuss, 2009; 1% contaminant, Ratcliff & Tuerlinckx, 2002).
All log-likelihoods on the scale of t in seconds; PSIS-LOO over responses. T4 GPU → Run all. Resumable.

In [ ]:
!pip -q install numpyro multipledispatch --no-deps
import os, time, json, numpy as np, pandas as pd, jax, jax.numpy as jnp, numpyro, numpyro.distributions as dist, optax, arviz as az
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoNormal
import statsmodels.api as sm
from google.colab import drive
drive.mount('/content/drive'); assert jax.devices()[0].platform=='gpu', 'Select T4 GPU'
S2='/content/drive/MyDrive/PISA2025_stage2'; OUT='/content/drive/MyDrive/PISA2025_stage3'; os.makedirs(OUT,exist_ok=True)
CNTS=['ARM','KAZ','MDA','LVA','LTU','EST','JPN','ARE','SGP']; SEED=20251025; STEPS=15000; NDRAW=200; EPS=0.01

D=pd.read_parquet(f'{S2}/sample_math.parquet')
ii,items=pd.factorize(D.item); pp,pers=pd.factorize(D.CNTSTUID)
pc=D.drop_duplicates('CNTSTUID').set_index('CNTSTUID').loc[pers]; cp=pd.Categorical(pc.CNT,categories=CNTS).codes
ts=D.t.values/10.                                                        # time unit = 10 s for the diffusion model
g=pd.Series(ts).groupby(ii)
d=dict(ii=jnp.array(ii),pp=jnp.array(pp),cp=jnp.array(cp),x=jnp.array(D.x.values,jnp.float32),lt=jnp.array(np.log(D.t.values),jnp.float32),
       ts=jnp.array(ts,jnp.float32),mint=jnp.array(g.min().values,jnp.float32),tmax=jnp.array(g.max().values,jnp.float32))
nI,nP,nC=len(items),len(pers),len(CNTS); print(nI,nP,len(D))

def eta_of(p,cp,fix1):
    mu=jnp.concatenate([jnp.zeros((1,2)),p['mu_free']],0); sd=p['sd'].at[0].set(1.) if fix1 else p['sd']
    return mu[cp]+p['z']@(sd[:,None]*p['Lcorr']).T

def wfpt(t,v,a,w):                                                       # log density, lower boundary (Navarro & Fuss 2009)
    u=t/a**2; k=jnp.arange(1,21.)
    sl=jnp.sum(k*jnp.sin(k*jnp.pi*w)*jnp.exp(-(k**2-1)*jnp.pi**2*u[...,None]/2),-1)
    large=jnp.log(jnp.pi)+jnp.log(jnp.maximum(sl,1e-20))-jnp.pi**2*u/2
    ww=w+2*jnp.arange(-5,6.); ss=jnp.sum(ww*jnp.exp(-(ww**2-w**2)/(2*u[...,None])),-1)
    small=-0.5*jnp.log(2*jnp.pi*u**3)+jnp.log(jnp.maximum(ss,1e-20))-w**2/(2*u)
    return jnp.where(u<0.5,small,large)-2*jnp.log(a)-v*a*w-v**2*t/2

def LL_vdl(p,d):
    e=eta_of(p,d['cp'],True); th,ta=e[:,0],e[:,1]; i,q=d['ii'],d['pp']
    lx=dist.Bernoulli(logits=p['a'][i]*(th[q]-p['b'][i])).log_prob(d['x'])
    return lx+dist.Normal(p['beta'][i]-ta[q],1./p['alpha'][i]).log_prob(d['lt'])-d['lt']
def kappa(p,d): return p['kap0']+jnp.concatenate([jnp.zeros(1),p['kap_free']])[d['cp']]+p['sk']*p['u']
def LL_mix(p,d):
    rg=dist.Bernoulli(probs=p['g']).log_prob(d['x'])+dist.Normal(p['mur'],p['sr']).log_prob(d['lt'])-d['lt']
    k=kappa(p,d)[d['pp']]
    return jnp.logaddexp(jax.nn.log_sigmoid(-k)+LL_vdl(p,d),jax.nn.log_sigmoid(k)+rg)
def LL_diff(p,d):
    e=eta_of(p,d['cp'],False); th,c=e[:,0],e[:,1]; i,q=d['ii'],d['pp']
    v=th[q]-p['b'][i]; a=jnp.exp(p['la'][i]+c[q]); ter=d['mint'][i]*jax.nn.sigmoid(p['r'][i])*0.99
    td=jnp.maximum(d['ts']-ter,1e-4)
    lw=jnp.where(d['x']>0.5,wfpt(td,-v,a,0.5),wfpt(td,v,a,0.5))
    lc=jnp.log(0.5)-jnp.log(d['tmax'][i])
    return jnp.logaddexp(jnp.log(1-EPS)+lw,jnp.log(EPS)+lc)-jnp.log(10.)
LL={'M1_vdl':LL_vdl,'M3_mix':LL_mix,'M4_diff':LL_diff}

def make_model(name):
    def model(d):
        S=numpyro.sample; p={}
        p['mu_free']=S('mu_free',dist.Normal(0.,1.).expand([nC-1,2])); p['Lcorr']=S('Lcorr',dist.LKJCholesky(2,2.))
        p['sd']=S('sd',dist.HalfNormal(1.).expand([2])); p['z']=S('z',dist.Normal(0.,1.).expand([nP,2]))
        if name in('M1_vdl','M3_mix'):
            p['a']=S('a',dist.LogNormal(0.,.5).expand([nI])); p['b']=S('b',dist.Normal(0.,2.).expand([nI]))
            p['beta']=S('beta',dist.Normal(4.,1.).expand([nI])); p['alpha']=S('alpha',dist.LogNormal(0.,.5).expand([nI]))
        if name=='M3_mix':
            p['kap0']=S('kap0',dist.Normal(-3.,1.)); p['kap_free']=S('kap_free',dist.Normal(0.,1.).expand([nC-1]))
            p['sk']=S('sk',dist.HalfNormal(1.)); p['u']=S('u',dist.Normal(0.,1.).expand([nP]))
            p['g']=S('g',dist.Beta(2.,6.)); p['mur']=S('mur',dist.Normal(1.,.5)); p['sr']=S('sr',dist.HalfNormal(.5))
        if name=='M4_diff':
            p['b']=S('b',dist.Normal(0.,2.).expand([nI])); p['la']=S('la',dist.Normal(1.,1.).expand([nI])); p['r']=S('r',dist.Normal(0.,1.5).expand([nI]))
        numpyro.factor('obs',LL[name](p,d).sum())
    return model

def run(name):
    if os.path.exists(f'{OUT}/{name}_res.json'): print(name,'already done'); return
    model=make_model(name); guide=AutoNormal(model,init_scale=0.05)
    opt=numpyro.optim.optax_to_numpyro(optax.chain(optax.clip_by_global_norm(10.),optax.adam(optax.exponential_decay(0.02,4000,0.5,end_value=0.001))))
    t0=time.time(); res=SVI(model,guide,opt,Trace_ELBO()).run(jax.random.PRNGKey(SEED),STEPS,d,progress_bar=True); rt=time.time()-t0
    Ls=np.asarray(res.losses); np.save(f'{OUT}/{name}_elbo.npy',Ls)
    s=Predictive(guide,params=res.params,num_samples=NDRAW)(jax.random.PRNGKey(1),d); s={k:v for k,v in s.items() if k!='obs'}
    ll=jax.lax.map(lambda j: LL[name](jax.tree_util.tree_map(lambda a:a[j],s),d),jnp.arange(NDRAW)); ll=np.asarray(ll)
    idata=az.from_dict(posterior={'dummy':np.zeros((1,NDRAW))},log_likelihood={'obs':ll[None]}); idata.to_netcdf(f'{OUT}/{name}_ll.nc')
    lo=az.loo(idata)
    fix1=name!='M4_diff'; E=np.asarray(jax.vmap(lambda j: eta_of(jax.tree_util.tree_map(lambda a:a[j],s),d['cp'],fix1))(jnp.arange(NDRAW)))
    P=pc[['CNT','AIUSESCH','ESCS','ST004D01T','EFFORT1','W_FSTUWT','ST438Q01DA','ST438Q02DA','ST438Q03DA','ST438Q04DA']].copy()
    P['p1']=E[...,0].mean(0); P['p1_sd']=E[...,0].std(0); P['p2']=E[...,1].mean(0); P['p2_sd']=E[...,1].std(0)
    if name=='M3_mix': P['p3']=np.asarray(jax.vmap(lambda j: kappa(jax.tree_util.tree_map(lambda a:a[j],s),d))(jnp.arange(NDRAW))).mean(0)
    P.reset_index().to_csv(f'{OUT}/{name}_persons.csv',index=False)
    it={k:np.asarray(s[k]).mean(0) for k in ['a','b','beta','alpha','la','r'] if k in s}; pd.DataFrame(it,index=items).to_csv(f'{OUT}/{name}_items.csv')
    mu=np.vstack([np.zeros((1,2)),np.asarray(s['mu_free']).mean(0)])
    out={'runtime_min':round(rt/60,1),'neg_elbo':round(float(Ls[-500:].mean()),1),'elbo_rel_change':round(float((Ls[-1500:-1000].mean()-Ls[-500:].mean())/abs(Ls[-500:].mean())),6),
         'elpd_loo':round(float(lo.elpd_loo),1),'se':round(float(lo.se),1),'p_loo':round(float(lo.p_loo),1),'pareto_k_gt_0.7_pct':round(100*float((lo.pareto_k.values>0.7).mean()),2),
         'corr_p1_p2':round(float(np.asarray(s['Lcorr'])[:,1,0].mean()),3),'mu_p1':dict(zip(CNTS,mu[:,0].round(3))),'mu_p2':dict(zip(CNTS,mu[:,1].round(3)))}
    if name=='M3_mix': out.update({'g':round(float(np.asarray(s['g']).mean()),3),'rapid_mean_sec':round(float(np.exp(np.asarray(s['mur']).mean())),2)})
    json.dump(out,open(f'{OUT}/{name}_res.json','w'),indent=1); print(name,json.dumps(out))

for m in ['M1_vdl','M3_mix','M4_diff']: run(m)

cmp=az.compare({m:az.from_netcdf(f'{OUT}/{m}_ll.nc') for m in LL},ic='loo'); cmp.to_csv(f'{OUT}/stage3_compare.csv'); print(cmp[['rank','elpd_loo','p_loo','elpd_diff','dse']].round(1))
rows=[]
for m,lab in [('M1_vdl',['ability','speed']),('M3_mix',['ability','speed','logit_rapid']),('M4_diff',['drift','log_caution'])]:
    P=pd.read_csv(f'{OUT}/{m}_persons.csv').dropna(subset=['AIUSESCH','ESCS','ST004D01T']); P['w']=P.W_FSTUWT/P.groupby('CNT').W_FSTUWT.transform('mean')
    X=sm.add_constant(pd.get_dummies(P[['AIUSESCH','ESCS','ST004D01T','CNT']],columns=['CNT','ST004D01T'],drop_first=True,dtype=float))
    for y,l in zip(['p1','p2','p3'],lab):
        r=sm.WLS(P[y],X,weights=P.w).fit(cov_type='HC1'); rows.append({'model':m,'param':l,'b_AI':r.params['AIUSESCH'],'se':r.bse['AIUSESCH'],'p':r.pvalues['AIUSESCH']})
R=pd.DataFrame(rows).round(4); R.to_csv(f'{OUT}/stage3_ai_preview.csv',index=False); print(R); print('STAGE3 DONE')
